# 00 — Verify Connection

Confirms Databricks Connect is working against Serverless compute and that the Unity Catalog target exists (or creates it).

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

# No cluster_id = Serverless
spark = DatabricksSession.builder.serverless().getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.1.0


In [2]:
# Confirm Spark is remote (not local)
spark.sql("SELECT current_user(), current_timestamp()").show(truncate=False)

+------------------------------+--------------------------+
|current_user()                |current_timestamp()       |
+------------------------------+--------------------------+
|alexander.booth@databricks.com|2026-03-09 20:45:23.489459|
+------------------------------+--------------------------+



In [3]:
UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "salt_river_excel")
UC_VOLUME  = os.getenv("UC_VOLUME",  "raw_files")

# Create schema and volume if they don't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}.{UC_VOLUME}")

print(f"Target: {UC_CATALOG}.{UC_SCHEMA}.{UC_VOLUME}")
spark.sql(f"DESCRIBE SCHEMA {UC_CATALOG}.{UC_SCHEMA}").show(truncate=False)

Target: alexander_booth.salt_river_excel.raw_files
+-------------------------+------------------------------+
|database_description_item|database_description_value    |
+-------------------------+------------------------------+
|Catalog Name             |alexander_booth               |
|Namespace Name           |salt_river_excel              |
|Comment                  |                              |
|Location                 |                              |
|Owner                    |alexander.booth@databricks.com|
+-------------------------+------------------------------+



In [4]:
# Upload the sample Excel file to the Volume
from databricks.sdk import WorkspaceClient
import pathlib

w = WorkspaceClient()

local_file = pathlib.Path("sample_data/salt_river_fields.xlsx")
volume_path = f"/Volumes/{UC_CATALOG}/{UC_SCHEMA}/{UC_VOLUME}/salt_river_fields.xlsx"

with open(local_file, "rb") as f:
    w.files.upload(volume_path, f, overwrite=True)

print(f"Uploaded to: {volume_path}")

Uploaded to: /Volumes/alexander_booth/salt_river_excel/raw_files/salt_river_fields.xlsx
